# 📈 Stock Market Prediction — PyTorch LSTM (Phase 1)

This notebook rebuilds the same multivariate LSTM from the TensorFlow v2 notebook in **PyTorch**.
Everything is kept identical for a fair comparison:
- Same features: `Close, Volume, RSI, MACD`
- Same lookback: `150 days`
- Same train/test split: `80/20`
- Same date range: `2012-01-01 → 2024-12-31`
- Same scaler: `MinMaxScaler(0.05, 0.95)` per feature

**Sections:**
1. Imports & Config
2. Data Download & Feature Engineering
3. Preprocessing & Scaling
4. Dataset & DataLoader
5. PyTorch LSTM Model
6. Training Loop
7. Evaluation & Metrics
8. TF vs PyTorch Comparison
9. Save PyTorch Model

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")

# ── Config (identical to TF v2 notebook) ─────────────────────────────────────
STOCK       = 'GOOG'
START       = '2012-01-01'
END         = '2024-12-31'
LOOKBACK    = 150
TRAIN_SPLIT = 0.80
FEATURES    = ['Close', 'Volume', 'RSI', 'MACD']
MODEL_PATH  = 'Stock_Predictions_PyTorch.pt'
# ─────────────────────────────────────────────────────────────────────────────

## 2. Data Download & Feature Engineering

Identical feature pipeline to TF v2 notebook.

In [ ]:
def compute_rsi(close, period=14):
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd_hist(close, fast=12, slow=26, signal=9):
    ema_f = close.ewm(span=fast,   adjust=False).mean()
    ema_s = close.ewm(span=slow,   adjust=False).mean()
    macd  = ema_f - ema_s
    sig   = macd.ewm(span=signal, adjust=False).mean()
    return macd - sig

# ── Download ──────────────────────────────────────────────────────────────────
raw = yf.download(STOCK, START, END, auto_adjust=True, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw.dropna(inplace=True)

# ── Features ──────────────────────────────────────────────────────────────────
data = raw.copy()
close = data['Close'].squeeze()
data['RSI']  = compute_rsi(close)
data['MACD'] = compute_macd_hist(close)
data = data[FEATURES].copy()
data.dropna(inplace=True)

print(f"Ticker  : {STOCK}")
print(f"Rows    : {len(data)}")
print(f"Range   : {data.index[0].date()} → {data.index[-1].date()}")
data.tail(3)

## 3. Preprocessing & Scaling

Per-feature `MinMaxScaler(0.05, 0.95)` — identical to TF v2. Fit on train only.

In [ ]:
feature_data = data.values.astype(np.float32)
split        = int(len(feature_data) * TRAIN_SPLIT)
train_raw    = feature_data[:split]
test_raw     = feature_data[split:]

scalers      = {}
train_scaled = np.zeros_like(train_raw, dtype=np.float32)
test_scaled  = np.zeros_like(test_raw,  dtype=np.float32)

for i, feat in enumerate(FEATURES):
    sc = MinMaxScaler(feature_range=(0.05, 0.95))
    train_scaled[:, i] = sc.fit_transform(train_raw[:, i].reshape(-1, 1)).flatten()
    test_scaled[:, i]  = sc.transform(test_raw[:, i].reshape(-1, 1)).flatten()
    scalers[feat] = sc

close_scaler = scalers['Close']

print(f"Train rows : {len(train_scaled)}")
print(f"Test rows  : {len(test_scaled)}")
print("Feature ranges (should all be 0.05–0.95):")
for i, f in enumerate(FEATURES):
    print(f"  {f:10s} [{train_scaled[:,i].min():.3f}, {train_scaled[:,i].max():.3f}]")

## 4. Dataset & DataLoader

PyTorch uses `Dataset` and `DataLoader` classes instead of raw numpy arrays.
- `Dataset` — defines how to get one sample `(X, y)`
- `DataLoader` — batches samples, shuffles, and feeds them to the model

In [ ]:
class StockDataset(Dataset):
    """PyTorch Dataset for time-series sequences."""

    def __init__(self, scaled_data: np.ndarray, lookback: int):
        self.X, self.y = [], []
        for i in range(lookback, len(scaled_data)):
            self.X.append(scaled_data[i - lookback:i, :])   # (lookback, n_features)
            self.y.append(scaled_data[i, 0])                 # Close only
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):              return len(self.y)
    def __getitem__(self, idx):     return self.X[idx], self.y[idx]


# Prepend last LOOKBACK train rows to give test sequences context
test_ctx      = np.concatenate([train_scaled[-LOOKBACK:], test_scaled], axis=0)

train_dataset = StockDataset(train_scaled, LOOKBACK)
test_dataset  = StockDataset(test_ctx,     LOOKBACK)

# Validation split from training data (last 10%)
val_size      = int(0.1 * len(train_dataset))
train_size    = len(train_dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds,    batch_size=16, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,      batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,batch_size=16, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")
print(f"Sample X shape: {train_dataset[0][0].shape}  → (lookback, n_features)")

## 5. PyTorch LSTM Model

**Key PyTorch vs TensorFlow differences:**

| Concept | TensorFlow/Keras | PyTorch |
|---|---|---|
| Model definition | `Sequential([...])` | `class Model(nn.Module)` |
| Forward pass | Automatic | You define `forward()` explicitly |
| Training loop | `model.fit()` | Manual loop: zero_grad → forward → loss → backward → step |
| Hidden state | Managed internally | You pass `(h0, c0)` explicitly |
| Input shape | `(batch, timesteps, features)` | `(batch, timesteps, features)` (same with `batch_first=True`) |

In [ ]:
class StockLSTM(nn.Module):
    """
    4-layer stacked LSTM matching the TF v2 architecture:
    units: 50 → 60 → 80 → 120, with dropout between each layer.
    """

    def __init__(self, input_size: int, hidden_sizes: list, dropout_rates: list):
        super(StockLSTM, self).__init__()

        self.lstms   = nn.ModuleList()
        self.drops   = nn.ModuleList()
        self.n_layers = len(hidden_sizes)

        in_size = input_size
        for h, d in zip(hidden_sizes, dropout_rates):
            # batch_first=True → input shape: (batch, seq_len, features)
            self.lstms.append(nn.LSTM(in_size, h, batch_first=True))
            self.drops.append(nn.Dropout(d))
            in_size = h

        self.fc = nn.Linear(hidden_sizes[-1], 1)

    def forward(self, x):
        """
        x: (batch, lookback, n_features)
        All intermediate LSTM layers return the full sequence (for next layer).
        Final LSTM layer returns only the last timestep's output.
        """
        out = x
        for i, (lstm, drop) in enumerate(zip(self.lstms, self.drops)):
            # h0, c0 default to zeros — PyTorch handles this automatically
            out, _ = lstm(out)                    # (batch, seq_len, hidden)
            if i < self.n_layers - 1:
                out = drop(out)                   # keep full sequence for next layer
            else:
                out = drop(out[:, -1, :])         # last timestep only → (batch, hidden)
        return self.fc(out)                       # (batch, 1)


# ── Instantiate — mirrors TF v2 units exactly ─────────────────────────────────
model = StockLSTM(
    input_size   = len(FEATURES),
    hidden_sizes = [50, 60, 80, 120],
    dropout_rates= [0.2, 0.3, 0.4, 0.5]
).to(DEVICE)

criterion  = nn.MSELoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-3)
# ReduceLROnPlateau — equivalent to TF's ReduceLROnPlateau callback
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-7, verbose=True
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

## 6. Training Loop

PyTorch doesn't have `model.fit()` — you write the training loop manually.
This gives you full control over every step:
```
for each epoch:
    for each batch:
        1. optimizer.zero_grad()   — clear previous gradients
        2. output = model(X)       — forward pass
        3. loss = criterion(...)   — compute loss
        4. loss.backward()         — backpropagation
        5. optimizer.step()        — update weights
```

In [ ]:
EPOCHS        = 100
PATIENCE      = 10       # EarlyStopping patience

train_losses  = []
val_losses    = []
best_val_loss = float('inf')
patience_ctr  = 0
best_epoch    = 0

for epoch in range(EPOCHS):

    # ── Training phase ───────────────────────────────────────────────────────
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()                          # 1. clear gradients
        preds = model(X_batch).squeeze(-1)             # 2. forward pass
        loss  = criterion(preds, y_batch)              # 3. compute loss
        loss.backward()                                # 4. backprop
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clip
        optimizer.step()                               # 5. update weights
        batch_losses.append(loss.item())

    train_loss = np.mean(batch_losses)

    # ── Validation phase ─────────────────────────────────────────────────────
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            preds   = model(X_batch).squeeze(-1)
            val_batch_losses.append(criterion(preds, y_batch).item())

    val_loss = np.mean(val_batch_losses)
    scheduler.step(val_loss)   # reduce LR if plateau

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # ── EarlyStopping + ModelCheckpoint ──────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        patience_ctr  = 0
        torch.save(model.state_dict(), MODEL_PATH)   # save best weights
        saved = '✅ saved'
    else:
        patience_ctr += 1
        saved = ''

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"train_loss: {train_loss:.6f} | "
              f"val_loss: {val_loss:.6f} {saved}")

    if patience_ctr >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}. "
              f"Best epoch: {best_epoch+1} (val_loss={best_val_loss:.6f})")
        break

print(f"\n✅ Best model saved → '{MODEL_PATH}'")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(train_losses, label='Train Loss', color='blue',   lw=1.5)
axes[0].plot(val_losses,   label='Val Loss',   color='orange', lw=1.5)
axes[0].axvline(best_epoch, color='red', ls='--', lw=1.2,
                label=f'Best epoch ({best_epoch+1})')
axes[0].set_title('PyTorch LSTM — Loss Over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Zoom in: skip first epoch (high loss) for better readability
axes[1].plot(train_losses[1:], label='Train Loss', color='blue',   lw=1.5)
axes[1].plot(val_losses[1:],   label='Val Loss',   color='orange', lw=1.5)
axes[1].set_title('Loss (zoomed — epoch 2 onwards)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation & Metrics

Load the best saved weights, run inference on the test set, inverse-transform to USD.

In [ ]:
# Load best weights
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        preds   = model(X_batch).squeeze(-1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())

y_pred_scaled = np.array(all_preds)
y_true_scaled = np.array(all_true)

# Inverse-transform to USD
y_pred_real = close_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true_real = close_scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()

# Align dates
test_dates = data.index[split:]
n = min(len(y_pred_real), len(y_true_real), len(test_dates))
y_pred_real = y_pred_real[:n]
y_true_real = y_true_real[:n]
test_dates  = test_dates[:n]

In [ ]:
mae  = mean_absolute_error(y_true_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
mape = np.mean(np.abs((y_true_real - y_pred_real) / y_true_real)) * 100
r2   = 1 - np.sum((y_true_real - y_pred_real)**2) / \
           np.sum((y_true_real - np.mean(y_true_real))**2)

print("═" * 42)
print(f"  MAE   : ${mae:.4f}")
print(f"  RMSE  : ${rmse:.4f}")
print(f"  MAPE  : {mape:.2f}%")
print(f"  R²    : {r2:.4f}")
print("═" * 42)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_true_real, color='green', lw=1.5, label='Actual Price')
plt.plot(test_dates, y_pred_real, color='red',   lw=1.5, ls='--', label='Predicted (PyTorch)')
plt.fill_between(test_dates,
                 y_pred_real * 0.97,
                 y_pred_real * 1.03,
                 color='red', alpha=0.1, label='±3% Band')
plt.title(f'{STOCK} — Actual vs Predicted (PyTorch LSTM)')
plt.xlabel('Date'); plt.ylabel('Price (USD)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. TensorFlow vs PyTorch Comparison

Update `TF_METRICS` below with the values from your TF v2 notebook before running.

In [ ]:
# ── Paste your TF v2 metrics here ─────────────────────────────────────────────
TF_METRICS = {
    'MAE':  5.6272,    # from TF v2 notebook
    'RMSE': 7.1158,
    'MAPE': 4.27,
    'R2':   0.9400
}

PT_METRICS = {
    'MAE':  mae,
    'RMSE': rmse,
    'MAPE': mape,
    'R2':   r2
}

print("\n" + "═" * 52)
print(f"{'Metric':<10} {'TensorFlow v2':>18} {'PyTorch':>18}")
print("─" * 52)
for k in ['MAE', 'RMSE', 'MAPE', 'R2']:
    tf_v = TF_METRICS[k]
    pt_v = PT_METRICS[k]
    better = '← better' if (
        (k in ['MAE','RMSE','MAPE'] and pt_v < tf_v) or
        (k == 'R2' and pt_v > tf_v)
    ) else ''
    unit = '%' if k == 'MAPE' else ('$' if k != 'R2' else '')
    print(f"  {k:<8} {unit}{tf_v:>16.4f}   {unit}{pt_v:>16.4f}  {better}")
print("═" * 52)

# Side-by-side prediction chart
print("\n(Re-run TF v2 notebook prediction to overlay — see chart below)")

## 9. Save PyTorch Model

Two save formats:
- **`state_dict`** — just the weights (recommended, portable, already saved during training)
- **`TorchScript`** — full model + weights, deployable without Python class definition (needed for FastAPI)

In [ ]:
# 1. State dict (already saved during training as best model)
torch.save(model.state_dict(), MODEL_PATH)
print(f"✅ State dict → '{MODEL_PATH}'")

# 2. TorchScript — needed for FastAPI deployment (Phase 2)
# Traces the model with a dummy input → self-contained, no class needed at load time
dummy_input   = torch.randn(1, LOOKBACK, len(FEATURES)).to(DEVICE)
scripted_model = torch.jit.trace(model, dummy_input)
scripted_model.save('Stock_Predictions_PyTorch_scripted.pt')
print(f"✅ TorchScript → 'Stock_Predictions_PyTorch_scripted.pt'")

# 3. Save scalers for use in FastAPI
import pickle
with open('pytorch_scalers.pkl', 'wb') as f:
    pickle.dump({'scalers': scalers, 'features': FEATURES,
                 'lookback': LOOKBACK, 'split': split}, f)
print(f"✅ Scalers     → 'pytorch_scalers.pkl'")

print()
print(f"   Architecture : 4-layer Stacked LSTM (50→60→80→120)")
print(f"   Input shape  : (batch, {LOOKBACK}, {len(FEATURES)})")
print(f"   Features     : {FEATURES}")
print(f"   MAE / RMSE / MAPE / R² : ${mae:.2f} / ${rmse:.2f} / {mape:.2f}% / {r2:.4f}")
print()
print("These 3 files are needed for Phase 2 (FastAPI):")
print("  1. Stock_Predictions_PyTorch_scripted.pt")
print("  2. pytorch_scalers.pkl")
print("  3. Stock Predictions Model.keras  (TF model for comparison endpoint)")